# Person 2: Location & geography -> `fac_geo`

**Owns:** red flags **B1-B13** and **J1-J2** from `RED_FLAGS_AND_VALIDATION.md`.
**Delivers:** the `fac_geo` table, the user-input geocoder (`geocoder.py`), and the district/state coverage tables.

Everything lives in `Person_2/`. The shared `Dataset/` folder is never modified; this notebook reads the copies in `Person_2/data/`.

| Step | Section | Flags |
|---|---|---|
| India Post reference (clean PIN centroids) | 1 | B12, B13 |
| State normalisation and recovery | 3 | B1, B2, B5 |
| PIN validation and extraction | 4 | B3, B4, B10 |
| Coordinates vs PIN | 5 | B6, B7, B8, B9 |
| City aliases and city-vs-PIN | 6 | B11 |
| Location decision rule, trust and OSM check | 7, 7b | section 2.2 of the playbook, L3 |
| Coverage / data density | 9 | J1, J2 |
| Geocoder for user input | 10 | B11 (user side) |

**Contract:** keyed by `unique_id`; booleans are `flag_<name>`; one `reasons` column (JSON list of sentences); raw values kept as `raw_<col>`.

## 0. Setup

In [ ]:
import sys, json, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 100)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

HERE = Path(".").resolve()
if not (HERE / "geo_pipeline.py").exists() and (HERE / "Person_2" / "geo_pipeline.py").exists():
    HERE = HERE / "Person_2"
sys.path.insert(0, str(HERE))
import geo_pipeline as gp
import geocoder as G

DATA = HERE / "data" if (HERE / "data" / "facilities.csv").exists() else HERE.parent / "Dataset"      # local copies, else the shared Dataset/ (read-only use)
OUT = HERE / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
def savefig(name): plt.savefig(OUT / "figures" / name, dpi=130, bbox_inches="tight")
print("working in", HERE)

## 1. India Post reference (B12, B13)
The directory is the yardstick for every location check, so it has to be clean first.
**B12:** 7% of offices have no coordinates and thousands are garbage (outside India). We keep an office coordinate only if it is inside the India bounding box **and** within 1000 km of its own state's median, then take the **median per PIN**.
**B13:** some PINs span two districts (district-by-PIN is approximate) and 52 span two states.

In [ ]:
t0 = time.time()
P1_BASE = HERE.parent / "Person_1" / "output" / "fac_base.parquet"      # Person 1's table (needs: pip install pyarrow)
if P1_BASE.exists():
    base = pd.read_parquet(P1_BASE)
    info = {"base": "Person 1 fac_base.parquet", "rows": len(base), "columns": base.shape[1]}
else:
    base, info = gp.load_base(DATA / "facilities.csv")                    # stand-in: no stitching of the 30 truncated records
    info["base"] = "stand-in (Person 1 fac_base not found)"
ref = gp.build_reference(DATA / "india_post_pincode_directory.csv")
print("base:", info, "| reference built in %.1fs" % (time.time() - t0))
pd.Series(ref["stats"], name="value").to_frame()

In [ ]:
pr = ref["ref"]
print("PINs:", len(pr), "| centroid from office coordinates:", int((pr.centroid_source == "offices").sum()), "| from district centroid (fallback):", int((pr.centroid_source == "district").sum()))
print("spread of offices around their PIN centroid (km): median %.1f | p90 %.1f | p99 %.1f -> a PIN centroid is accurate to a few km" % tuple(pr.spread_km.quantile(q) for q in (.5, .9, .99)))
print("B13: PINs spanning >1 district: %d (%.1f%%) | spanning >1 state: %d" % (pr.spans_districts.sum(), pr.spans_districts.mean() * 100, pr.spans_states.sum()))
print("\nPINs that span two states (first 8):"); print(pr[pr.spans_states][["state", "district", "n_offices", "n_states"]].head(8).to_string())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(pr.spread_km.dropna().clip(upper=30), bins=60, color="#4C72B0"); ax[0].set_title("Median distance of a PIN's offices to its centroid (km, clipped 30)")
s = ref["stats"]; ax[1].bar(["usable", "outside India", "far from state", "no coords"], [s["offices_usable"], s["offices_outside_india"], s["offices_far_from_state"], s["offices_missing_coords"]], color=["#55A868", "#C44E52", "#DD8452", "#8172B2"])
ax[1].set_title("India Post office coordinates (B12)"); ax[1].set_yscale("log"); plt.tight_layout(); savefig("01_reference_quality.png"); plt.show()

## 2. Build `fac_geo`
One call runs every step below. The sections after this one just inspect the result flag by flag.

In [ ]:
osm_path = OUT / "osm_health_poi.csv"          # produced once by fetch_osm.py (OpenStreetMap hospitals/clinics/doctors/dentists)
osm = pd.read_csv(osm_path) if osm_path.exists() else None
print("OpenStreetMap places loaded:", 0 if osm is None else len(osm))
t0 = time.time()
geo = gp.build_fac_geo(base, ref, osm=osm)
print(geo.shape, "built in %.1fs" % (time.time() - t0), "| one row per unique_id:", geo.unique_id.is_unique, "| rows == base:", len(geo) == len(base))
flags = [c for c in geo.columns if c.startswith("flag_")]
ft = geo[flags].sum().sort_values(ascending=False).to_frame("facilities"); ft["%"] = (ft.facilities / len(geo) * 100).round(1); ft

## 3. State: normalise, recover, resolve conflicts (B1, B2, B5)
* **B1** the state field is spelled in hundreds of ways -> alias map + India Post canonical names.
* **B2** city or district typed in the state field -> take the state from the PIN.
* **B5** state vs PIN-state: **AP/Telangana** (2014 split) always trust the PIN. For other conflicts: if the coordinates agree with the address state, the *PIN* is the odd one out (a directory error such as Agartala PIN 799001 filed under Telangana), so it is ignored (`flag_pin_state_unreliable`); otherwise the PIN wins.

In [ ]:
raw_n = geo.raw_stateOrRegion.nunique(); print("distinct raw state strings:", raw_n, "-> distinct canonical states after normalisation:", geo.state_norm.nunique(), "(India has 28 states + 8 UTs = 36)")
print("B2: state field is not a state:", int(geo.flag_state_field_not_a_state.sum()), "| recovered from PIN:", int(geo.flag_state_recovered_from_pin.sum()),
      "| still unknown after PIN:", int((geo.flag_state_field_not_a_state & geo.pin_state.isna()).sum()), "| final state missing:", int(geo.state_final.isna().sum()))
print("\nexamples of non-state values in the state field:"); print(geo[geo.flag_state_field_not_a_state].raw_stateOrRegion.value_counts().head(12).to_dict())
print("\nstate_source:"); print(geo.state_source.value_counts().to_string())

In [ ]:
c = geo[geo.flag_state_conflict]
print("B5: state vs PIN-state conflicts:", len(c), "| AP<->Telangana:", int(c.flag_state_conflict_ap_telangana.sum()), "| other:", int(c.flag_state_conflict_other.sum()))
print("resolved how:", c.state_source.value_counts().to_dict())
print("\nAll 'other' conflicts, with the evidence used:")
oth = c[c.flag_state_conflict_other].merge(base[["unique_id", "name", "address_city"]], on="unique_id")
oth[["name", "address_city", "state_norm", "pin6", "pin_state_listed", "coord_state", "state_final", "state_source", "flag_pin_state_unreliable"]].head(20)

## 4. PIN codes: clean, validate, extract (B3, B4, B10)
Strip spaces/hyphens (`"201 301"`, `"641-602"`); if the field is unusable, read a PIN out of the address lines; flag PINs that are not in the directory; flag addresses that mention **two or more different valid PINs** (a merge signal).

In [ ]:
print("PIN field:  reformatted (space/hyphen removed):", int(geo.flag_pin_reformatted.sum()), "| malformed (not 6 digits even after cleaning):", int(geo.flag_pin_malformed.sum()),
      "| 6 digits but NOT in India Post:", int(geo.flag_pin_invalid.sum()))
print("pin_source:", geo.pin_source.value_counts().to_dict())
print("valid PIN (in directory) after cleaning:", int(geo.pin_valid.sum()), "of", len(geo), f"({geo.pin_valid.mean():.1%})")
raw_ok = geo.raw_zipOrPostcode.fillna("").str.fullmatch("[1-9][0-9]{5}") & geo.raw_zipOrPostcode.fillna("").isin(set(ref["ref"].index))
print("before cleaning that number was", int(raw_ok.sum()), f"({raw_ok.mean():.1%})  -> cleaning recovered", int(geo.pin_valid.sum() - raw_ok.sum()), "PINs")
print("\nB10: addresses mentioning >=2 different valid PINs:", int(geo.flag_multiple_pins.sum()), "| across different states:", int(geo.flag_multiple_pin_states.sum()))
print("(the playbook reports 196; it appears to count PIN-like numbers elsewhere in the record too. This count uses the address fields only, which is the stricter definition.)")
print("\nmalformed examples:", geo[geo.flag_pin_malformed].raw_zipOrPostcode.head(10).tolist())
m = geo[geo.flag_multiple_pins].merge(base[["unique_id", "name", "address_city"]], on="unique_id")
m[["name", "address_city", "pin6", "n_distinct_pins", "flag_multiple_pin_states"]].head(8)

## 5. Coordinates vs the PIN centroid (B6, B7, B8, B9)
* **B8** outside the India bounding box -> NULL (original kept in `raw_latitude/longitude`; swapped lat/lon would be fixed).
* **B6 / B7** distance to the PIN centroid: > 25 km = conflict, > 100 km = almost certainly wrong or merged.
* **B9** many facilities on the identical point -> a locality-level geocode, so exact-distance ranking is not meaningful.

In [ ]:
print(geo.geo_status.value_counts().to_string())
d = geo.geo_mismatch_km.dropna()
print("\ndistance coordinates -> PIN centroid (km): median %.1f | p75 %.1f | p90 %.0f | p99 %.0f" % tuple(d.quantile(q) for q in (.5, .75, .9, .99)))
print({f">{t} km": int((d > t).sum()) for t in (10, 25, 50, 100, 500, 1000)})
print("B8: coordinates outside India (nulled):", int(geo.flag_coords_outside_india.sum()), "| swapped & fixed:", int(geo.flag_latlon_swapped.sum()))
print("B9: facilities on a shared point:", int(geo.flag_shared_coords.sum()), "| distinct shared points:", int(geo[geo.flag_shared_coords].groupby(["lat", "lon"]).ngroups))
print("coordinates state != resolved state (nearest PIN <50 km away):", int(geo.flag_coord_state_mismatch.sum()))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.log10(d + 0.1), bins=70, color="#4C72B0"); [ax[0].axvline(np.log10(t), color="r", ls="--") for t in (25, 100)]; ax[0].set_xlabel("log10(km from PIN centroid)"); ax[0].set_title("Coordinates vs PIN centroid")
geo.geo_status.value_counts().plot.barh(ax=ax[1], color="#DD8452"); ax[1].set_title("geo_status"); plt.tight_layout(); savefig("02_coordinates_vs_pin.png"); plt.show()
x = geo[geo.flag_geo_mismatch_100km].merge(base[["unique_id", "name", "address_city"]], on="unique_id").sort_values("geo_mismatch_km", ascending=False)
x[["name", "address_city", "pin6", "pin_district", "coord_district", "geo_mismatch_km"]].head(10)

In [ ]:
sh = geo[geo.flag_shared_coords].merge(base[["unique_id", "name", "address_city"]], on="unique_id")
top = sh.groupby(["lat", "lon"]).agg(n=("unique_id", "size"), city=("address_city", "first"), names=("name", lambda s: "; ".join(s.head(3).str[:26]))).sort_values("n", ascending=False).head(8)
print("largest shared points:"); top

## 6. City names (B11)
Old and new names (Bangalore/Bengaluru, Gurgaon/Gurugram, Trivandrum/Thiruvananthapuram, ...) are resolved with one alias table that is shared with the user-input geocoder.
`city_matches_pin` compares the stated city with the PIN's district and post-office names.

In [ ]:
print("city alias entries:", len(gp.CITY_ALIASES), "| e.g.", dict(list({k: v for k, v in gp.CITY_ALIASES.items() if k in ("bangalore", "gurgaon", "trivandrum", "bombay", "calcutta", "poona", "cochin", "allahabad")}.items())))
k = geo.city_matches_pin.dropna().astype(bool)
print(f"stated city matches the PIN's district or a post-office name: {k.mean():.1%} of {len(k)}")
print("\nmatch rate by location status (city agreement should track the other signals):")
print(geo.dropna(subset=["city_matches_pin"]).groupby("location_trust").city_matches_pin.apply(lambda s: round(s.astype(bool).mean(), 3)).to_string())
mm = geo[geo.flag_city_mismatch].merge(base[["unique_id", "name"]], on="unique_id")
mm[["name", "raw_city", "pin6", "pin_district", "location_trust"]].sample(10, random_state=3)

## 7. Location decision rule and trust (playbook section 2.2)
| Case | Location used | `location_trust` | `location_precision` |
|---|---|---|---|
| coordinates within 25 km of the PIN centroid | coordinates | `verified` | `coordinates` (or `locality` if shared) |
| conflict, an OpenStreetMap place matches near one side | that side | `verified_osm` | |
| conflict, **same district as the PIN and within 100 km** (*added: large districts*) | coordinates | `probable` | `coordinates` |
| conflict, city matches the PIN's area | PIN centroid | `approximate_pin` | `pin_centroid` |
| conflict, nothing agrees | PIN centroid | `unverified` | `pin_centroid` |
| only a PIN | PIN centroid | `approximate_pin` | `pin_centroid` |
| only coordinates | coordinates | `unverified` | |
| neither, but the district is known | district centre | `approximate_district` | `district_centroid` |

`location_trust_score` is a 0-1 score: base by status (verified_osm 1.0, verified 0.90, probable 0.75, approximate_pin 0.60, approximate_district 0.40, unverified 0.25), minus small penalties for shared coordinates, several PINs, state conflict, district conflict, city mismatch, or a PIN that spans districts.

In [ ]:
print(geo.location_trust.value_counts().to_string()); print()
print(geo.location_precision.value_counts().to_string()); print()
print(geo.location_trust_level.value_counts().reindex(["high", "medium", "low"]).to_string())
print("\nlocation_trust: mean %.2f | median %.2f" % (geo.location_trust_score.mean(), geo.location_trust_score.median()))
print("facilities with a usable best location:", int(geo.best_lat.notna().sum()), "of", len(geo))

### 7b. OpenStreetMap corroboration (L3)
A facility is matched to an OSM place when the names are at least 80% similar and the place is **within 2 km of the coordinates or of the PIN centroid**. When coordinates and PIN disagree, the side the OSM place supports wins (`verified_osm`). One-off download by `fetch_osm.py` from the public Overpass server.

In [ ]:
if osm is None:
    print("osm_health_poi.csv not found: run fetch_osm.py once (see README). The osm_* columns stay empty.")
else:
    print("OSM places:", len(osm), "|", osm.amenity.value_counts().to_dict())
    print("facilities with a matching OSM place:", int(geo.osm_match.eq(True).sum()), f"({geo.osm_match.eq(True).mean():.1%}) | median distance {geo.osm_dist_km.median():.2f} km")
    x = geo[geo.osm_match.notna()].assign(osm=lambda d: d.osm_match.astype(bool))
    print("\nshare matched, by geo_status (OSM coverage is incomplete, so a miss is not evidence of a wrong location):"); print(x.groupby("geo_status").osm.agg(["mean", "size"]).round(3).to_string())
    c = geo[geo.geo_status == "conflict"]
    print(f"\nOf {len(c)} coordinate-vs-PIN conflicts: OSM supports the COORDINATES for {int((c.osm_near == 'coords').sum())}, the PIN for {int((c.osm_near == 'pin').sum())}, neither/no match for {int(c.osm_near.isna().sum())}")
    ev = c[c.osm_near.notna()].assign(winner=lambda d: np.where(d.osm_near == "coords", "coordinates", "PIN centroid"))
    print("-> where OSM can arbitrate, the coordinates are right %.0f%% of the time, so 'ignore coordinates when they disagree with the PIN' would have discarded good data" % (100 * (ev.winner == "coordinates").mean()))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    geo.location_trust.value_counts().plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("location_trust (with OSM)")
    pd.Series({"coordinates": int((c.osm_near == "coords").sum()), "PIN": int((c.osm_near == "pin").sum()), "no OSM match": int(c.osm_near.isna().sum())}).plot.bar(ax=ax[1], color=["#2a9d8f", "#e9c46a", "#bbbbbb"])
    ax[1].set_title("Conflicts: what does OSM support?"); plt.tight_layout(); savefig("03b_osm_arbitration.png"); plt.show()

In [ ]:
col = {"high": "#2a9d8f", "medium": "#e9c46a", "low": "#e63946"}
fig, ax = plt.subplots(1, 2, figsize=(13, 6.4))
pl = geo.dropna(subset=["best_lat"])
for lab in ["high", "medium", "low"]:
    s = pl[pl.location_trust_level == lab]; ax[0].scatter(s.best_lon, s.best_lat, s=3, c=col[lab], alpha=.6, label=f"{lab} ({len(s)})")
ax[0].legend(markerscale=4); ax[0].set_aspect("equal"); ax[0].set_title("Best location, by location trust")
hb = ax[1].hexbin(pl.best_lon, pl.best_lat, gridsize=45, cmap="viridis", mincnt=1, bins="log"); ax[1].set_aspect("equal"); ax[1].set_title("Facility density (log)"); fig.colorbar(hb, ax=ax[1], shrink=.6)
plt.tight_layout(); savefig("03_map_location_trust.png"); plt.show()
top = geo.state_final.value_counts().head(14).index
tab = pd.crosstab(geo.state_final, geo.location_trust).loc[top]
tab.div(tab.sum(axis=1), axis=0).plot.barh(stacked=True, figsize=(9, 5), colormap="Set2"); plt.gca().invert_yaxis(); plt.title("Location status by state (share)"); plt.legend(fontsize=7, bbox_to_anchor=(1, 1)); plt.tight_layout(); savefig("04_status_by_state.png"); plt.show()

In [ ]:
# What the app can show: a plain-language reason list per facility
pick = pd.concat([geo[geo.location_trust == s].sample(1, random_state=7) for s in ["verified", "probable", "approximate_pin", "unverified"]])
pick = pick.merge(base[["unique_id", "name", "address_city"]], on="unique_id")
for _, r in pick.iterrows():
    print(f"[{r.location_trust} | trust {r.location_trust_score} | {r.location_precision}] {r['name'][:50]} ({r.address_city})")
    for s in json.loads(r.reasons): print("   -", s)

## 8. Does the PIN-based correction actually improve things? (sanity checks)

In [ ]:
import re
checks = {
 "unique_id is unique and complete": geo.unique_id.is_unique and len(geo) == len(base),
 "every best location is inside India": bool(gp.in_india(geo.best_lat.dropna().values, geo.best_lon.dropna().values).all()),
 "trust is within [0,1]": bool(geo.location_trust_score.between(0, 1).all()),
 "every row has at least one reason": bool((geo.reasons.map(lambda s: len(json.loads(s))) >= 1).all()),
 "'verified' implies <=25 km (or OSM)": bool((geo[geo.location_trust == "verified"].geo_mismatch_km <= 25).all()),
 "no coordinates kept outside India": bool(gp.in_india(geo.lat.dropna().values, geo.lon.dropna().values).all()),
 "state_final is a canonical state or missing": bool(geo.state_final.dropna().isin(set(ref["canon"])).all()),
 "raw_ columns preserved (same non-null count as base)": bool(geo.raw_zipOrPostcode.notna().sum() == base.address_zipOrPostcode.notna().sum()),
}
pd.Series(checks, name="passed").to_frame()

In [ ]:
# Compare with the counts quoted in RED_FLAGS_AND_VALIDATION.md (differences are explained, not hidden)
doc = pd.DataFrame([
 ("B2  state field holds a city/district", 284, int(geo.flag_state_field_not_a_state.sum()), "alias map also reads 'Thane, Maharashtra' style values"),
 ("B3  PIN has a space/hyphen (reformatted)", 153, int(geo.flag_pin_reformatted.sum()), ""),
 ("B3  PIN malformed", 70, int(geo.flag_pin_malformed.sum()), ""),
 ("B4  PIN not in India Post", 215, int(geo.flag_pin_invalid.sum()), "PINs recovered from the address text no longer count"),
 ("B5  state vs PIN conflict (real)", 65, int(geo.flag_state_conflict.sum()), "AP/Telangana = %d; %d PINs distrusted (directory error)" % (int(geo.flag_state_conflict_ap_telangana.sum()), int(geo.flag_pin_state_unreliable.sum()))),
 ("B6  coords > 25 km from PIN", 1965, int(geo.flag_geo_mismatch_25km.sum()), "cleaner PIN centroids + recovered PINs"),
 ("B7  coords > 100 km from PIN", 1227, int(geo.flag_geo_mismatch_100km.sum()), "same"),
 ("B8  coords outside India", 6, int(geo.flag_coords_outside_india.sum()), ""),
 ("B9  facilities on shared coordinates", 79, int(geo.flag_shared_coords.sum()), ""),
 ("B10 address mentions >=2 valid PINs", 196, int(geo.flag_multiple_pins.sum()), "stricter: address fields only"),
 ("B12 garbage India Post coordinates", 2613, ref["stats"]["offices_outside_india"], "doc 2,613 includes far-from-state; we report both"),
 ("B13 PINs spanning two states", 52, ref["stats"]["pins_span_states"], ""),
], columns=["flag", "playbook count", "this notebook", "note"])
doc["difference"] = doc["this notebook"] - doc["playbook count"]; doc

### 8b. Cross-check against Person 1's own location columns
`fac_base` already carries a few location columns that Person 1 derived on the way (`pin6`, `state_norm`, `pin_state`, `city_matches_pin`, `km_coords_to_pin`). Two people computing the same thing independently is a useful test; differences show where one of us made a different choice.

In [ ]:
p1cols = ["pin6", "pin_state", "state_norm", "pin_valid", "city_matches_pin", "km_coords_to_pin"]
if all(c in base.columns for c in p1cols):
    x = base[["unique_id"] + p1cols].merge(geo[["unique_id", "pin6", "pin_state", "state_norm", "pin_valid", "city_matches_pin", "geo_mismatch_km"]], on="unique_id", suffixes=("_p1", "_p2"))
    same = lambda a, b: round(float((x[a].fillna("~").astype(str) == x[b].fillna("~").astype(str)).mean()), 4)
    tab = pd.Series({c: same(c + "_p1", c + "_p2") for c in ["pin6", "pin_state", "state_norm", "pin_valid", "city_matches_pin"]}, name="share identical").to_frame()
    d = (x.km_coords_to_pin - x.geo_mismatch_km).abs()
    print("km to PIN centroid: median abs difference %.2f km | rows that differ by >5 km: %d (P1 has a distance for %d facilities, this notebook for %d)" % (d.median(), int((d > 5).sum()), int(x.km_coords_to_pin.notna().sum()), int(x.geo_mismatch_km.notna().sum())))
    ov = sorted((set(base.columns) & set(geo.columns)) - {"unique_id"})
    print("column names that exist in BOTH fac_base and fac_geo (prefix or drop one side in the final join):", ov)
    tab
else:
    print("Person 1's location columns are not in this base: skipping the cross-check")

## 9. Coverage and data density (J1, J2)
The UI must say *"few facilities in our data for this area"*. Every district gets a facility count, a density class, and a ready-to-show message. NFHS districts are matched to India Post districts (renames such as Bangalore -> Bengaluru Urban are handled by an alias table).

In [ ]:
nf, nf_map, nf_un = gp.map_nfhs(ref, DATA / "nfhs_5_district_health_indicators.csv")
print("NFHS districts matched to an India Post district:", len(nf) - len(nf_un), "of", len(nf), "| unmatched:", nf_un[["state", "district_name"]].values.tolist())
dens, st = gp.build_density(geo, ref, nf, nf_map, base=base)
print("\nIndia Post districts:", len(dens), "| with zero facilities:", int((dens.n_facilities == 0).sum()), "| sparse (1-3):", int((dens.data_density == "sparse").sum()),
      "| moderate:", int((dens.data_density == "moderate").sum()), "| dense:", int((dens.data_density == "dense").sum()))
print("NFHS districts with zero facilities in our data:", int(((dens.n_facilities == 0) & dens.nfhs_matched).sum()), "(each has an NFHS health profile but no facility to recommend)")
print("\nJ1: top-5 states hold %.1f%% of facilities; states/UTs with <30 facilities:" % st.share_pct.head(5).sum(), st[st.n_facilities < 30].state_final.tolist())
st.head(12)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dd = dens.n_facilities.clip(upper=60); ax[0].hist(dd, bins=61, color="#4C72B0"); ax[0].set_title("Facilities per district (clipped at 60)"); ax[0].set_xlabel("facilities")
s = st.sort_values("n_facilities").tail(18); ax[1].barh(s.state_final.str.title(), s.n_facilities, color="#55A868"); ax[1].tick_params(axis="y", labelsize=7); ax[1].set_title("Facilities per state / UT")
plt.tight_layout(); savefig("05_coverage.png"); plt.show()
dens[dens.data_density.isin(["none", "sparse"])].sort_values("n_pins", ascending=False)[["state", "district", "n_pins", "n_facilities", "data_density", "coverage_message"]].head(10)

## 10. User-input geocoder
Accepts a city, district, PIN, `lat, lon`, or `city, state`; applies the alias table; asks for the state when a name is ambiguous; falls back to fuzzy matching for typos. Everything runs on India Post + `fac_geo` (no internet).

In [ ]:
gc = G.Geocoder(ref, geo)
tests = [("Bangalore", None), ("Gurgaon", None), ("800 001", None), ("Patna", None), ("25.59, 85.13", None), ("Aurangabad", None), ("Aurangabad", "Bihar"), ("Jaipur, Rajasthan", None),
         ("Trivandrum", None), ("Banglore", None), ("Benaras", None), ("Hamirpur", None), ("Uttar Pradesh", None), ("Leh", None), ("999999", None), ("Xyzzy", None)]
rows = []
for q, s_ in tests:
    r = gc.geocode(q, state=s_)
    if r["status"] == "ok": rows.append((q, s_, "ok", r["level"], f"{r['lat']:.3f}, {r['lon']:.3f}", f"{r['state']} / {r['district']}", r["confidence"], r["note"] or ""))
    elif r["status"] == "ambiguous": rows.append((q, s_, "ambiguous", "", "", "; ".join(f"{c['district']} ({c['state']})" for c in r["candidates"]), "", r["message"]))
    else: rows.append((q, s_, "not_found", "", "", "", "", r["message"]))
pd.DataFrame(rows, columns=["query", "state hint", "status", "level", "lat, lon", "state / district", "confidence", "note"])

In [ ]:
r = gc.geocode("Patna"); near, info = gc.within_radius(r["lat"], r["lon"], 25, geo)
print("Patna, 25 km:", info["n_facilities"], "facilities in our data | trust mix:", near.location_trust_level.value_counts().to_dict())
r2 = gc.geocode("Tawang, Arunachal Pradesh"); print("Tawang:", r2.get("status"), r2.get("district"))
if r2["status"] == "ok":
    n2, i2 = gc.within_radius(r2["lat"], r2["lon"], 50, geo); print("Tawang, 50 km:", i2)
print("reverse geocode of (25.60, 85.14):", gc.reverse(25.60, 85.14))

## 10b. Contract check
The revised playbook fixes the names other people will read. This cell asserts them, so a rename can never silently break Person 3 or the final ranking.
`location_trust` is the label from the decision rule. This notebook adds two tiers to the playbook's five (`probable`, `approximate_district`); the numeric score is `location_trust_score`.

In [ ]:
need = ["unique_id", "pin6", "pin_district", "pin_state", "state_final", "geo_mismatch_km", "geo_status", "city_matches_pin", "osm_match", "best_lat", "best_lon",
        "location_precision", "location_trust", "reasons", "addr_consistent", "flag_pin_invalid", "flag_state_conflict", "flag_multiple_pins", "flag_shared_coords"]
core = {"verified", "verified_osm", "approximate_pin", "unverified", "unknown"}
res = {
 "all contract columns present": all(c in geo.columns for c in need),
 "geo_status uses the contract values": set(geo.geo_status.unique()) <= {"consistent", "conflict", "coords_only", "pin_only", "invalid"},
 "location_trust labels beyond the playbook's five": sorted(set(geo.location_trust.unique()) - core),
 "reasons is a JSON list of strings": all(isinstance(json.loads(x), list) for x in geo.reasons),
 "every flag_ column is boolean": all(geo[c].dtype == bool for c in geo.columns if c.startswith("flag_")),
 "pin_reference has lat, lon, district, state": {"lat", "lon", "district", "state"} <= set(ref["ref"].columns),
 "geocode_user_input('Bangalore')": G.geocode_user_input("Bangalore", geocoder=gc),
 "geocode_user_input('800001')": G.geocode_user_input("800001", geocoder=gc),
 "geocode_user_input('Aurangabad')": G.geocode_user_input("Aurangabad", geocoder=gc),
}
print("geo_status:", geo.geo_status.value_counts().to_dict()); print("location_trust:", geo.location_trust.value_counts().to_dict())
pd.Series(res, name="result").to_frame()

## 11. Outputs
Written to `Person_2/outputs/`. CSV is used because the local machine has no Parquet engine; the Databricks snippet at the bottom loads them as Delta tables.

In [ ]:
DESC = {
 "unique_id": "key; matches fac_base", "raw_stateOrRegion": "original state field", "raw_zipOrPostcode": "original PIN field", "raw_city": "original city",
 "raw_latitude": "original latitude (text)", "raw_longitude": "original longitude (text)", "state_norm": "state read from the address (canonical India Post name) or null",
 "flag_state_field_not_a_state": "B2: state field held a city/district or junk", "flag_pin_reformatted": "B3: PIN had a space/hyphen and was cleaned", "flag_pin_malformed": "B3: PIN not 6 digits even after cleaning",
 "flag_pin_invalid": "B4: 6-digit PIN not in India Post", "pin6": "best PIN (field, cleaned, or read from address text)", "pin_source": "field | address_text | field_not_in_directory | none",
 "pin_valid": "pin6 exists in India Post and is not distrusted", "addr_consistent": "all valid PINs in the address point to one district (null if no valid PIN)", "pin_state_listed": "state the directory gives for pin6, kept only when the PIN was distrusted", "flag_pin_state_unreliable": "address state AND coordinates both disagree with the PIN's state: the PIN is ignored (directory error)", "n_distinct_pins": "distinct valid PINs found in the address", "flag_multiple_pins": "B10: address mentions 2+ valid PINs", "flag_multiple_pin_states": "B10: those PINs are in different states",
 "pin_state": "state of pin6 (India Post)", "pin_district": "district of pin6", "pin_dkey": "normalised district key (join key)", "pin_lat": "clean PIN centroid latitude", "pin_lon": "clean PIN centroid longitude",
 "pin_centroid_source": "offices | district (fallback)", "pin_spread_km": "median distance of the PIN's offices to its centroid", "flag_pin_spans_districts": "B13: PIN covers more than one district",
 "flag_pin_spans_states": "B13: PIN covers more than one state", "flag_latlon_swapped": "B8: lat/lon swapped and corrected", "flag_coords_outside_india": "B8: coordinates outside India (nulled)",
 "lat": "coordinates kept after B8 (null if rejected)", "lon": "coordinates kept after B8 (null if rejected)", "coord_pin": "nearest PIN to the coordinates", "coord_state": "state of that PIN",
 "coord_district": "district of that PIN", "coord_dkey": "normalised district key of that PIN", "coord_nearest_pin_km": "distance to that PIN centroid", "geo_mismatch_km": "B6/B7: km from coordinates to the stated PIN's centroid",
 "flag_geo_mismatch_25km": "B6: more than 25 km", "flag_geo_mismatch_100km": "B7: more than 100 km", "geo_status": "consistent | conflict | coords_only | pin_only | invalid (contract values)",
 "shared_coord_n": "number of facilities on the identical point", "flag_shared_coords": "B9: coordinates shared with another facility", "geocode_precision": "coordinates | locality | none",
 "flag_state_conflict": "B5: address state differs from PIN state", "flag_state_conflict_ap_telangana": "B5: the AP/Telangana case (trust the PIN)", "flag_state_conflict_other": "B5: any other conflict",
 "state_final": "resolved state (PIN > coordinates > address)", "state_source": "how state_final was chosen", "flag_state_recovered_from_pin": "B2: state taken from the PIN because the field was not a state",
 "flag_coord_state_mismatch": "coordinates fall in a different state than state_final", "district_final_key": "normalised district key (join to district tables)", "district_final": "district name (India Post)",
 "district_state": "state of that district", "district_source": "pin | coords | none", "flag_district_conflict": "PIN district differs from the coordinates' district", "city_key": "normalised city (aliases applied)",
 "city_matches_pin": "stated city matches the PIN's district/post offices (null if unknown)", "flag_city_mismatch": "stated city does not match the PIN's area", "osm_match": "L3: matching OSM place (null until OSM is loaded)",
 "osm_dist_km": "L3: distance to that OSM place", "osm_near": "L3: coords | pin (which side the OSM place supports)", "best_lat": "location to use for distance ranking", "best_lon": "location to use for distance ranking",
 "location_precision": "coordinates | locality | pin_centroid | district_centroid | none", "location_trust": "verified | verified_osm | probable | approximate_pin | approximate_district | unverified | unknown",
 "location_trust_score": "0-1 score (see section 7); location_trust_level = low/medium/high", "location_trust_level": "low | medium | high", "reasons": "JSON list of plain-language sentences for the UI",
}
missing = [c for c in geo.columns if c not in DESC]; extra = [c for c in DESC if c not in geo.columns]
print("columns without a description:", missing, "| descriptions for unknown columns:", extra)
geo_out = geo.copy(); geo_out["location_trust_level"] = geo_out.location_trust_level.astype(str)
geo_out.to_csv(OUT / "fac_geo.csv", index=False)
pd.DataFrame({"column": list(geo.columns), "description": [DESC.get(c, "") for c in geo.columns]}).to_csv(OUT / "fac_geo_columns.csv", index=False)
ref["ref"].reset_index().rename(columns={"index": "pincode"}).to_csv(OUT / "pin_reference.csv", index=False)
dens.drop(columns=["nfhs_row"], errors="ignore").assign(data_density=dens.data_density.astype(str)).to_csv(OUT / "district_density.csv", index=False)
st.to_csv(OUT / "state_coverage.csv", index=False)
pd.DataFrame({"alias": list(gp.CITY_ALIASES), "canonical": list(gp.CITY_ALIASES.values())}).to_csv(OUT / "city_alias.csv", index=False)
nf[["state", "district_name", "dkey_matched"]].to_csv(OUT / "nfhs_district_map.csv", index=False)
print({p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in sorted(OUT.glob("*.csv"))})

In [ ]:
# Databricks Free Edition: upload the CSVs to a Volume, then
#   spark.read.option("header", True).option("multiLine", True).option("escape", chr(34)).csv("/Volumes/<catalog>/<schema>/<volume>/fac_geo.csv") \
#        .write.mode("overwrite").saveAsTable("<catalog>.<schema>.fac_geo")
# The booleans arrive as strings ("True"/"False"); cast them with F.col(c).cast("boolean") for every column starting with flag_.
print("fac_geo.csv rows:", len(pd.read_csv(OUT / "fac_geo.csv", usecols=["unique_id"])))

In [ ]:
# data_quality_report.md, section 2 (generated from this run, so the numbers cannot drift from the data)
NL = chr(10)
n = len(geo); r = ref["stats"]
rows = [
 ("B1", "State field has hundreds of spellings", f"{geo.raw_stateOrRegion.nunique()} strings -> {geo.state_norm.nunique()} canonical", "FIX: alias map + India Post names"),
 ("B2", "City/district typed in the state field", int(geo.flag_state_field_not_a_state.sum()), f"FIX: state from the PIN ({int(geo.flag_state_recovered_from_pin.sum())} recovered)"),
 ("B3", "PIN had a space/hyphen / was malformed", f"{int(geo.flag_pin_reformatted.sum())} / {int(geo.flag_pin_malformed.sum())}", "FIX: cleaned; else read from the address text"),
 ("B4", "PIN not in the India Post directory", int(geo.flag_pin_invalid.sum()), "FLAG: flag_pin_invalid; falls back to coordinates or district"),
 ("B5", "State differs from the PIN's state", f"{int(geo.flag_state_conflict.sum())} ({int(geo.flag_state_conflict_ap_telangana.sum())} AP/Telangana)", f"FLAG; AP/Telangana trust the PIN; {int(geo.flag_pin_state_unreliable.sum())} PINs ignored where address and coordinates both disagree"),
 ("B6", "Coordinates > 25 km from the PIN centroid", int(geo.flag_geo_mismatch_25km.sum()), "FLAG: geo_mismatch_km, geo_status = conflict"),
 ("B7", "Coordinates > 100 km from the PIN", int(geo.flag_geo_mismatch_100km.sum()), "FLAG + trust penalty; PIN centroid used unless OSM or district agreement supports the coordinates"),
 ("B8", "Coordinates outside India", int(geo.flag_coords_outside_india.sum()), "NULL (original kept in raw_latitude / raw_longitude)"),
 ("B9", "Facilities on identical coordinates", int(geo.flag_shared_coords.sum()), "FLAG: location_precision = locality"),
 ("B10", "Address mentions 2+ distinct valid PINs", int(geo.flag_multiple_pins.sum()), "FLAG: flag_multiple_pins (feeds the merge score)"),
 ("B11", "City name variants", f"{len(gp.CITY_ALIASES)} aliases", "FIX: shared alias table, also used for user input"),
 ("B12", "India Post coordinates missing / garbage", f"{r['offices_missing_coords']} / {r['offices_outside_india'] + r['offices_far_from_state']}", "FIX: bounding box + state filter, median per PIN"),
 ("B13", "PINs spanning two districts / two states", f"{r['pins_span_districts']} / {r['pins_span_states']}", "FLAG: flag_pin_spans_districts / flag_pin_spans_states"),
 ("J1", "Top-5 states share of facilities", f"{st.share_pct.head(5).sum():.1f}%", "state_coverage table; UI coverage note"),
 ("J2", "Districts with no facilities (India Post / NFHS-linked)", f"{int((dens.n_facilities == 0).sum())} of {len(dens)} / {int(((dens.n_facilities == 0) & dens.nfhs_matched).sum())}", "district_density table"),
]
lines = ["## 2. Location and geography (Person 2)", "",
         f"Base: {n:,} facilities. Locations were decided with the rule in the playbook plus two additions (see Notes). Table: `fac_geo` (one row per `unique_id`).", "",
         "| # | Red flag | Count | Action |", "|---|---|---|---|"]
lines += [f"| {a} | {b} | {c} | {d} |" for a, b, c, d in rows]
tl = geo.location_trust.value_counts()
lines += ["", "### Outcome", "", "| location_trust | facilities | share |", "|---|---|---|"]
lines += [f"| {k} | {int(v)} | {v / n:.1%} |" for k, v in tl.items()]
lines += ["", f"Trust level (score-based): high {int((geo.location_trust_level == 'high').sum())}, medium {int((geo.location_trust_level == 'medium').sum())}, low {int((geo.location_trust_level == 'low').sum())}.",
          f"A usable best location exists for {int(geo.best_lat.notna().sum()):,} of {n:,}; precision: " + ", ".join(f"{k} {int(v)}" for k, v in geo.location_precision.value_counts().items()) + ".", "", "### Notes", ""]
if osm is not None:
    c = geo[geo.geo_status == "conflict"]
    lines += [f"* OpenStreetMap ({len(osm):,} places) matched {int(geo.osm_match.eq(True).sum()):,} facilities. Of {len(c)} coordinate-vs-PIN conflicts it supported the coordinates {int((c.osm_near == 'coords').sum())} times and the PIN {int((c.osm_near == 'pin').sum())} times."]
lines += ["* Added tier `probable`: coordinates in the same district as the PIN and within 100 km are kept. Where OSM could arbitrate, the coordinates were right about three times in four, so discarding them would lose good data.",
          "* Added tier `approximate_district` (only the district is known). `location_trust` therefore has seven labels, a superset of the playbook's five; map `probable` to verified-with-caution if a strict five-way label is needed.",
          "* The India Post directory is itself wrong for some PINs (for example 799001 in Agartala is filed under Telangana). When the address state and the coordinates both disagree with the PIN, the PIN is ignored.",
          "* B10 is counted on address fields only (stricter than the playbook's 196).",
          "* Built on Person 1's stitched `fac_base`: the 30 repaired records now carry coordinates and are located like every other facility.",
          "* District-by-PIN is approximate where a PIN spans districts; `flag_district_conflict` shows where the PIN and the coordinates disagree."]
(HERE / "data_quality_report_section2.md").write_text(NL.join(lines) + NL, encoding="utf-8")
print("wrote", HERE / "data_quality_report_section2.md", "|", len(lines), "lines")

## 12. Hand-offs and open items
* **Person 1:** done. This run uses P1's stitched `fac_base.parquet` (9,989 rows); the 30 repaired records now have coordinates (24 consistent with their PIN, 6 in conflict). If `fac_base` is re-issued, call `gp.build_fac_geo(base=fac_base, reference=ref, osm=osm)` again; it only needs the columns in `gp.REQUIRED`.
* **Final join:** `fac_base` and `fac_geo` share column names (see section 8b, including `reasons`). Prefix one side before joining the four tables.
* **Person 3 / 4:** join on `unique_id`. Use `location_trust_score` / `location_trust_level` in `facility_trust`, `best_lat/best_lon` for distance, and `flag_multiple_pins`, `flag_geo_mismatch_100km`, `flag_shared_coords` as inputs to `merge_suspect_score` (B6/B10 are the two location inputs in the playbook formula).
* **OpenStreetMap corroboration (L3):** done. `fetch_osm.py` downloaded 87,469 named places for 35 of 36 states/UTs (Lakshadweep failed; it has no facilities in the data). Re-running it is not needed unless the data changes. See section 7b.
* **District matching** is approximate where a PIN spans districts (`flag_pin_spans_districts`) and where district boundaries were redrawn; `flag_district_conflict` shows where the PIN and the coordinates disagree.